# 🤖 Agentic AI — Complete Guide
## Theory → Implementation → Sample Project (Research & Math Agent)

---

> **Agentic AI** is a system where a Large Language Model (LLM) autonomously **reasons**, **plans**, **calls tools**, **observes results**, and **iterates** to complete complex goals — without human intervention at every step.

### 📚 What You'll Learn
- ✅ Core concepts: Agents, Tools, Memory, ReAct
- ✅ How LangChain agents work internally
- ✅ Building tools from scratch
- ✅ Complete project: Research + Math Assistant Agent
- ✅ Multi-tool agent with memory
- ✅ Tracing agent reasoning step-by-step

---

### ⚡ Free API Options
| Provider | Model | Free Tier | Get Key |
|----------|-------|-----------|--------|
| **Google Gemini** | gemini-1.5-flash | 15 req/min | [aistudio.google.com](https://aistudio.google.com/apikey) |
| **Groq** | llama-3.1-8b | 30 req/min | [console.groq.com](https://console.groq.com) |
| **OpenAI** | gpt-4o-mini | Paid (~$0.001/req) | [platform.openai.com](https://platform.openai.com) |

## 📖 Section 1: Theory — How Agentic AI Works

### 1.1 The Agent Architecture

```
                    ┌─────────────────────────────────┐
  User Query ──────►│           LLM Agent              │
                    │   (Reasoning / Planning Engine)   │
                    └────────────────┬────────────────┘
                                     │ decides action
                    ┌────────────────▼────────────────┐
                    │          Tool Registry            │
                    │  • Wikipedia Search              │
                    │  • Calculator                    │
                    │  • Python REPL                   │
                    │  • Web Search                    │
                    │  • Database Query                │
                    └────────────────┬────────────────┘
                                     │ returns observation
                    ┌────────────────▼────────────────┐
                    │           Memory                  │
                    │  • Short-term (chat history)     │
                    │  • Long-term (vector DB)         │
                    └────────────────┬────────────────┘
                                     │ loop until done
                              Final Answer
```

### 1.2 The ReAct Framework (Reason + Act)

ReAct is the most popular agent paradigm, combining **reasoning** and **acting**:

```
Question: What is the capital of France and what is 1234 × 5678?

Thought 1: I need to find the capital of France and do a multiplication.
Action 1: Wikipedia Search('capital of France')
Observation 1: Paris is the capital city of France.

Thought 2: Now I need to calculate 1234 × 5678.
Action 2: Calculator('1234 * 5678')
Observation 2: 7,006,652

Thought 3: I have all the information needed.
Final Answer: The capital of France is Paris. 1234 × 5678 = 7,006,652
```

### 1.3 Agent vs Simple LLM

| Feature | Simple LLM | Agent |
|---------|-----------|-------|
| Input | Single prompt | Goal + conversation |
| Tools | ❌ None | ✅ Many tools |
| Memory | ❌ Stateless | ✅ Persistent memory |
| Iteration | ❌ One response | ✅ Multi-step loop |
| Real-time info | ❌ Training cutoff | ✅ Web search |
| Code execution | ❌ | ✅ Python REPL |

### 1.4 Key Components

1. **LLM (Brain)** — Decides WHAT to do and HOW to use tools
2. **Tools** — Functions the agent can call (search, calculate, etc.)
3. **Memory** — Stores conversation history and past observations
4. **Agent Executor** — Runs the Thought→Action→Observation loop
5. **Parser** — Extracts tool names and arguments from LLM output

## ⚙️ Section 2: Install Dependencies

In [ ]:
# Install required packages
!pip install -q langchain langchain-community langchain-google-genai
!pip install -q wikipedia-api numexpr google-generativeai
!pip install -q requests beautifulsoup4

print('✅ All packages installed!')
print()
print('Packages installed:')
print('  • langchain          — Agent framework & orchestration')
print('  • langchain-community — Tool integrations (Wikipedia, etc.)')
print('  • langchain-google-genai — Gemini LLM integration')
print('  • wikipedia-api      — Wikipedia search tool')
print('  • numexpr            — Fast mathematical expressions')
print('  • google-generativeai — Google Gemini SDK')

## 🔑 Section 3: API Key Setup

**Get a FREE Gemini API key:** https://aistudio.google.com/apikey

Or use **Groq** (also free): https://console.groq.com

> ⚠️ **Never share your API key or commit it to GitHub!**

In [ ]:
import os
import getpass

# ── Option 1: Google Gemini (RECOMMENDED — Free tier available) ──
print('Enter your API key below.')
print('Get FREE Gemini key: https://aistudio.google.com/apikey')
print('Get FREE Groq key  : https://console.groq.com')
print()

# Securely input the API key (won't show in output)
GOOGLE_API_KEY = getpass.getpass('Enter your Google Gemini API key: ')
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

print('✅ API key set!')
print(f'   Key preview: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}')

## 📦 Section 4: Imports & LLM Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# LangChain core
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import Tool
from langchain.memory import ConversationBufferMemory
from langchain import hub
from langchain.prompts import PromptTemplate

# LLM — Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Built-in tools
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Standard libraries
import math
import json
import requests
import re
import numexpr

print('✅ All imports successful!')

# ── Initialize LLM ────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model='gemini-1.5-flash',       # Fast, free model
    temperature=0,                   # 0 = deterministic for agents
    google_api_key=GOOGLE_API_KEY
)

# Test the LLM
test_response = llm.invoke('Hello! In one sentence, what is an AI agent?')
print('\nLLM Test Response:')
print(f'  {test_response.content}')
print('\n✅ LLM connected and working!')

## 🛠️ Section 5: Define Agent Tools

Tools are **functions** that the agent can call. Each tool has:
- `name` — Identifier the LLM uses to call it
- `description` — Tells the LLM WHEN and HOW to use it
- `func` — The actual Python function to execute

In [ ]:
# ─────────────────────────────────────────────────────────────
#  TOOL 1: Wikipedia Search
# ─────────────────────────────────────────────────────────────
wikipedia_api = WikipediaAPIWrapper(
    top_k_results=2,       # Return top 2 results
    doc_content_chars_max=1500  # Limit characters to avoid token overflow
)

def wikipedia_search(query: str) -> str:
    """Search Wikipedia for factual information."""
    try:
        result = wikipedia_api.run(query)
        return result if result else 'No results found on Wikipedia for this query.'
    except Exception as e:
        return f'Wikipedia search failed: {str(e)}. Try rephrasing your query.'

tool_wikipedia = Tool(
    name='Wikipedia',
    description=(
        'Search Wikipedia for factual information about people, places, events, '
        'science, history, or general knowledge. '
        'Input: a search query string. '
        'Use this when you need encyclopedic or factual information.'
    ),
    func=wikipedia_search
)

# ─────────────────────────────────────────────────────────────
#  TOOL 2: Calculator (Safe Math Evaluator)
# ─────────────────────────────────────────────────────────────
def calculator(expression: str) -> str:
    """
    Evaluate mathematical expressions safely.
    Uses numexpr for safe evaluation (no arbitrary code execution).
    """
    try:
        # Clean the expression
        expr = expression.strip()
        expr = expr.replace('^', '**')  # Support ^ for power
        expr = expr.replace('×', '*').replace('÷', '/')  # Unicode operators
        expr = expr.replace(',', '')    # Remove thousands separators
        
        # Use numexpr for safe evaluation
        result = numexpr.evaluate(expr)
        result = float(result)
        
        # Format output nicely
        if result == int(result) and abs(result) < 1e15:
            return f'{int(result):,}'
        else:
            return f'{result:,.6g}'
    except Exception as e:
        # Fallback to Python eval with restricted builtins
        try:
            safe_globals = {'__builtins__': {}}
            safe_locals = {
                'abs': abs, 'round': round, 'pow': pow,
                'sqrt': math.sqrt, 'log': math.log, 'log10': math.log10,
                'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
                'pi': math.pi, 'e': math.e, 'factorial': math.factorial,
                'ceil': math.ceil, 'floor': math.floor
            }
            result = eval(expr, safe_globals, safe_locals)
            if isinstance(result, (int, float)):
                if isinstance(result, int):
                    return f'{result:,}'
                return f'{result:,.6g}'
            return str(result)
        except Exception:
            return f'Could not evaluate: {expression}. Please use valid math notation like: 2+3*4, sqrt(16), 10^3'

tool_calculator = Tool(
    name='Calculator',
    description=(
        'Perform mathematical calculations. Supports: +, -, *, /, ** (power), '
        'sqrt(), log(), sin(), cos(), factorial(), pi, e. '
        'Input: a mathematical expression as a string. '
        'Example inputs: "1234 * 5678", "sqrt(144)", "2**10", "log(100)". '
        'Use this for ANY numeric calculation — never try to calculate in your head.'
    ),
    func=calculator
)

# ─────────────────────────────────────────────────────────────
#  TOOL 3: Unit Converter
# ─────────────────────────────────────────────────────────────
def unit_converter(query: str) -> str:
    """
    Convert between common units.
    Query format: 'VALUE UNIT to UNIT' (e.g., '100 km to miles')
    """
    conversions = {
        # Length
        'km_to_miles': 0.621371, 'miles_to_km': 1.60934,
        'meter_to_feet': 3.28084, 'feet_to_meter': 0.3048,
        'cm_to_inches': 0.393701, 'inches_to_cm': 2.54,
        # Weight
        'kg_to_pounds': 2.20462, 'pounds_to_kg': 0.453592,
        'gram_to_ounces': 0.035274, 'ounces_to_gram': 28.3495,
        # Temperature (handled specially)
        # Volume
        'liter_to_gallon': 0.264172, 'gallon_to_liter': 3.78541,
        'ml_to_cups': 0.00422675, 'cups_to_ml': 236.588,
        # Speed
        'kmh_to_mph': 0.621371, 'mph_to_kmh': 1.60934,
        'ms_to_kmh': 3.6, 'kmh_to_ms': 0.277778,
    }
    
    query = query.lower().strip()
    
    # Temperature conversion
    if 'celsius' in query and 'fahrenheit' in query:
        try:
            val = float(re.search(r'[\d.]+', query).group())
            if 'to fahrenheit' in query:
                result = (val * 9/5) + 32
                return f'{val}°C = {result:.2f}°F'
            else:
                result = (val - 32) * 5/9
                return f'{val}°F = {result:.2f}°C'
        except:
            pass
    
    # Other conversions
    for key, factor in conversions.items():
        from_unit, to_unit = key.split('_to_')
        if from_unit in query and to_unit in query:
            try:
                val = float(re.search(r'[\d.]+', query).group())
                result = val * factor
                return f'{val} {from_unit} = {result:.4f} {to_unit}'
            except:
                pass
    
    return f'Could not convert: {query}. Try format like: "100 km to miles", "75 kg to pounds", "100 celsius to fahrenheit"'

tool_converter = Tool(
    name='UnitConverter',
    description=(
        'Convert between units of measurement. '
        'Supports: length (km/miles/meter/feet/cm/inches), '
        'weight (kg/pounds/gram/ounces), '
        'temperature (Celsius/Fahrenheit), '
        'volume (liter/gallon/ml/cups), speed (kmh/mph/ms). '
        'Input format: "VALUE FROM_UNIT to TO_UNIT" '
        'Example: "100 km to miles", "75 kg to pounds", "100 celsius to fahrenheit"'
    ),
    func=unit_converter
)

# ─────────────────────────────────────────────────────────────
#  TOOL 4: Word Counter / Text Analyzer
# ─────────────────────────────────────────────────────────────
def text_analyzer(text: str) -> str:
    """Analyze text statistics."""
    words = text.split()
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    chars = len(text)
    chars_no_space = len(text.replace(' ', ''))
    unique_words = len(set(w.lower() for w in words))
    avg_word_len = sum(len(w) for w in words) / len(words) if words else 0
    
    return (
        f'Text Analysis Results:\n'
        f'  Words        : {len(words):,}\n'
        f'  Unique words : {unique_words:,}\n'
        f'  Sentences    : {len(sentences):,}\n'
        f'  Characters   : {chars:,}\n'
        f'  Chars (no sp): {chars_no_space:,}\n'
        f'  Avg word len : {avg_word_len:.1f} chars\n'
        f'  Reading time : ~{len(words)//200 + 1} minute(s) (200 wpm)'
    )

tool_text = Tool(
    name='TextAnalyzer',
    description=(
        'Analyze text to count words, characters, sentences, and get reading time. '
        'Input: the text string to analyze. '
        'Use this when asked to count words or analyze text properties.'
    ),
    func=text_analyzer
)

# ─────────────────────────────────────────────────────────────
#  Collect all tools
# ─────────────────────────────────────────────────────────────
tools = [tool_wikipedia, tool_calculator, tool_converter, tool_text]

print('🛠️  Agent Tools Ready:')
for t in tools:
    print(f'   ✅ {t.name}: {t.description[:60]}...')

## 🧪 Section 6: Test Each Tool Individually

In [ ]:
print('Testing each tool independently...')
print('─' * 50)

# Test Wikipedia
print('\n📚 Wikipedia Search:')
result = wikipedia_search('Artificial Intelligence history')
print(result[:300] + '...')

print('\n🔢 Calculator:')
tests = ['1234 * 5678', 'sqrt(144)', '2**10', '(100 + 50) * 3 / 2', 'factorial(10)']
for expr in tests:
    print(f'   {expr} = {calculator(expr)}')

print('\n📏 Unit Converter:')
conv_tests = [
    '100 km to miles',
    '75 kg to pounds', 
    '100 celsius to fahrenheit',
    '1 gallon to liter'
]
for q in conv_tests:
    print(f'   {q} → {unit_converter(q)}')

print('\n📝 Text Analyzer:')
sample = 'Artificial intelligence is the simulation of human intelligence in machines. It learns, reasons, and solves problems.'
print(text_analyzer(sample))

print('\n✅ All tools working correctly!')

## 🤖 Section 7: Build the ReAct Agent

We use the **ReAct** (Reason + Act) agent type which:
1. **Reasons** about what to do (Thought)
2. **Acts** by calling a tool (Action)
3. **Observes** the result (Observation)
4. Repeats until it has the final answer

In [ ]:
# ── Custom ReAct Prompt Template ──────────────────────────────
# This prompt tells the LLM HOW to behave as an agent
REACT_TEMPLATE = """You are a helpful AI Research Assistant with access to tools.
Always think step-by-step and use tools when needed instead of guessing.

You have access to the following tools:

{tools}

Use the following format EXACTLY:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

RULES:
- ALWAYS use Calculator for math (never calculate in your head)
- ALWAYS use Wikipedia for factual questions about people/places/events
- ALWAYS use UnitConverter for unit conversions
- ALWAYS use TextAnalyzer for word counting tasks
- Provide clear, concise final answers

Begin!

Previous conversation:
{chat_history}

Question: {input}
Thought: {agent_scratchpad}"""

react_prompt = PromptTemplate.from_template(REACT_TEMPLATE)

# ── Build the Agent ───────────────────────────────────────────
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt
)

# ── Memory ────────────────────────────────────────────────────
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=False
)

# ── Agent Executor ────────────────────────────────────────────
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,        # Show Thought/Action/Observation trace
    max_iterations=8,    # Max tool calls per query
    handle_parsing_errors=True,
    return_intermediate_steps=False
)

print('🤖 Agent built successfully!')
print('   Type: ReAct (Reason + Act)')
print('   LLM : Gemini 1.5 Flash')
print(f'   Tools: {[t.name for t in tools]}')
print('   Memory: ConversationBufferMemory')

## 🚀 Section 8: Run the Agent — Sample Queries

Watch the agent **think step-by-step** and use tools!

In [ ]:
def run_agent(query: str):
    """Run agent on a query and display the result cleanly."""
    print('\n' + '═' * 60)
    print(f'🧑 USER: {query}')
    print('═' * 60)
    
    response = agent_executor.invoke({'input': query})
    
    print('\n' + '─' * 60)
    print(f'🤖 FINAL ANSWER:')
    print(f"{response['output']}")
    print('─' * 60)
    return response['output']

# ── Query 1: Math Calculation ─────────────────────────────────
run_agent('What is 1234 multiplied by 5678? Then what is the square root of that result?')

In [ ]:
# ── Query 2: Wikipedia Search ─────────────────────────────────
run_agent('Who invented the World Wide Web and when was it invented?')

In [ ]:
# ── Query 3: Multi-tool (Search + Calculate) ──────────────────
run_agent(
    'Find the year Alan Turing was born. Then calculate how many years '
    'ago that was from 2024, and also convert 100 miles to kilometers.'
)

In [ ]:
# ── Query 4: Unit Conversion ──────────────────────────────────
run_agent('Convert 98.6 degrees Fahrenheit to Celsius. Is that normal human body temperature?')

In [ ]:
# ── Query 5: Memory Test (refers to previous answer) ──────────
run_agent('Based on our conversation, what calculations have we done so far?')

## 💬 Section 9: Interactive Agent Chat

Now you can chat with your agent interactively! Type questions and press Enter.

In [ ]:
print('💬 Interactive Agent Chat')
print('Type your question and press Enter. Type "quit" to stop.\n')
print('Example questions you can try:')
print('  • What is the speed of light in km/h?')
print('  • Who discovered penicillin and in what year?')
print('  • Convert 5000 rupees to... (just kidding, no currency tool yet!)')
print('  • What is 2 to the power of 32?')
print('  • Count the words in: The quick brown fox jumps over the lazy dog')
print()

while True:
    user_input = input('🧑 You: ').strip()
    
    if not user_input:
        continue
    
    if user_input.lower() in ['quit', 'exit', 'q', 'bye']:
        print('👋 Goodbye! The agent session has ended.')
        break
    
    try:
        response = agent_executor.invoke({'input': user_input})
        print(f'\n🤖 Agent: {response["output"]}\n')
        print('─' * 50)
    except Exception as e:
        print(f'❌ Error: {e}. Try rephrasing your question.\n')

## 🔧 Section 10: Building Your Own Custom Tool

Learn how to create custom tools and add them to your agent!

In [ ]:
# ── Example: Custom Fibonacci Tool ────────────────────────────
def fibonacci_tool(n_str: str) -> str:
    """Return the Nth Fibonacci number."""
    try:
        n = int(n_str.strip())
        if n < 0:
            return 'Please provide a non-negative integer.'
        if n > 100:
            return f'N={n} is too large. Please use N ≤ 100.'
        
        a, b = 0, 1
        for _ in range(n):
            a, b = b, a + b
        
        # Show first few Fibonacci numbers for context
        seq = [0, 1]
        for i in range(2, min(n + 1, 8)):
            seq.append(seq[-1] + seq[-2])
        
        return (
            f'Fibonacci({n}) = {a:,}\n'
            f'Sequence start: {seq}...'
        )
    except ValueError:
        return f'Invalid input: {n_str}. Please provide an integer like: 10'

tool_fibonacci = Tool(
    name='FibonacciCalculator',
    description=(
        'Calculate the Nth Fibonacci number. '
        'Input: a non-negative integer N (0 to 100). '
        'Returns the Nth Fibonacci number and sequence start. '
        'Use when asked about Fibonacci numbers.'
    ),
    func=fibonacci_tool
)

# ── Add to agent's toolset ─────────────────────────────────────
extended_tools = tools + [tool_fibonacci]

# Build new agent with extended tools
extended_agent = create_react_agent(
    llm=llm,
    tools=extended_tools,
    prompt=react_prompt
)

extended_executor = AgentExecutor(
    agent=extended_agent,
    tools=extended_tools,
    verbose=True,
    max_iterations=8,
    handle_parsing_errors=True,
    memory=ConversationBufferMemory(memory_key='chat_history', return_messages=False)
)

# Test the custom tool
print('Testing custom Fibonacci tool:')
print(fibonacci_tool('10'))
print()
print('Now running in agent:')
result = extended_executor.invoke({
    'input': 'What is the 15th Fibonacci number? '
             'Also, what is that number multiplied by itself?'
})
print(f'\n🤖 Answer: {result["output"]}')

## 📋 Section 11: Summary & Key Takeaways

### What We Built
| Component | Implementation |
|-----------|---------------|
| **LLM** | Google Gemini 1.5 Flash (free tier) |
| **Agent Type** | ReAct (Reason + Act) via LangChain |
| **Tools** | Wikipedia, Calculator, Unit Converter, Text Analyzer, Fibonacci |
| **Memory** | ConversationBufferMemory (short-term) |
| **Framework** | LangChain |

### ReAct Loop Recap
```
Query → Thought → Action → Observation → Thought → ... → Final Answer
```

### Key Concepts Covered
- 🔶 **Tools** = Python functions + description for LLM
- 🔶 **ReAct** = Reason + Act loop
- 🔶 **Memory** = Stores conversation context
- 🔶 **Agent Executor** = Runs the loop and stops at Final Answer
- 🔶 **Prompt Template** = Instructs LLM on HOW to behave

### Next Steps
- 🚀 Add **Web Search** tool (SerpAPI or DuckDuckGo)
- 🚀 Add **Long-term Memory** with ChromaDB vector store
- 🚀 Build a **Multi-Agent** system with LangGraph
- 🚀 Add **Code Interpreter** (PythonREPLTool)
- 🚀 Deploy as a **Streamlit** or **FastAPI** web app

### Useful Resources
- [LangChain Docs](https://python.langchain.com/docs/get_started/introduction)
- [LangGraph](https://github.com/langchain-ai/langgraph) — Stateful multi-step agents
- [Gemini API](https://aistudio.google.com) — Free LLM API
- [CrewAI](https://github.com/joaomdmoura/crewAI) — Multi-agent teams

In [ ]:
print('=' * 60)
print('       🤖  AGENTIC AI PROJECT COMPLETE  🤖')
print('=' * 60)
print()
print('Agent Configuration:')
print(f'  LLM         : Gemini 1.5 Flash')
print(f'  Agent type  : ReAct (Reason + Act)')
print(f'  Tools       : {[t.name for t in extended_tools]}')
print(f'  Memory      : ConversationBufferMemory')
print()
print('Concepts Mastered:')
print('  ✅ ReAct framework (Thought → Action → Observation)')
print('  ✅ Building custom tools with LangChain')
print('  ✅ Connecting LLMs to real-world functions')
print('  ✅ Agent memory for multi-turn conversations')
print('  ✅ Multi-tool agent orchestration')
print()
print('Next → Explore LangGraph for complex multi-agent systems!')
print('=' * 60)